In [1]:
import pandas as pd

In [2]:
visits = pd.read_csv('visits.csv')
registrations = pd.read_csv('registrations.csv')

In [5]:
visits.head()

,uuid,platform,user_agent,date
0,1de9ea66-70d3-4a1f-8735-df5ef7697fb9,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,2023-03-01T13:29:22
1,f149f542-e935-4870-9734-6b4501eaf614,web,Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...,2023-03-01T16:44:28
2,f149f542-e935-4870-9734-6b4501eaf614,web,Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...,2023-03-06T06:12:36
3,08f0ebd4-950c-4dd9-8e97-b5bdf073eed1,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,2023-03-01T20:16:37
4,08f0ebd4-950c-4dd9-8e97-b5bdf073eed1,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,2023-03-05T17:42:47


In [4]:
registrations.head()

,date,user_id,email,platform,registration_type
0,2023-03-01T00:25:39,8838849,joseph95@example.org,web,google
1,2023-03-01T14:53:01,8741065,janetsuarez@example.net,web,yandex
2,2023-03-01T14:27:36,1866654,robert67@example.org,web,google
3,2023-03-01T02:42:34,1577584,elam@example.net,web,apple
4,2023-03-01T10:27:14,4765395,stephanie68@example.net,web,yandex


In [13]:
visits = visits[~visits['user_agent'].str.contains('bot', case=False, na=False)].copy()

visits['date'] = pd.to_datetime(visits['date'])
registrations['date'] = pd.to_datetime(registrations['date'])

visits['date_group'] = visits['date'].dt.date
registrations['date_group'] = registrations['date'].dt.date

visits = visits.sort_values('date')

visits = visits.drop_duplicates(subset='uuid', keep='last')

visits_grouped = (
    visits
    .groupby(['date_group', 'platform'])
    .size()
    .reset_index(name='visits')
)

registrations_grouped = (
    registrations
    .groupby(['date_group', 'platform'])
    .size()
    .reset_index(name='registrations')
)

df = pd.merge(
    visits_grouped,
    registrations_grouped,
    on=['date_group', 'platform'],
    how='left'
)

df['registrations'] = df['registrations'].fillna(0)
df['conversion'] = (df['registrations'] / df['visits']) * 100
df = df.sort_values('date_group')

In [14]:
df.head()

,date_group,platform,visits,registrations,conversion
0,2023-03-01,android,27,21.0,77.777778
1,2023-03-01,ios,19,15.0,78.947368
2,2023-03-01,web,98,24.0,24.489796
3,2023-03-02,web,74,78.0,105.405405
4,2023-03-03,web,80,62.0,77.500000


In [15]:
df.to_json('conversion.json')